In [10]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [13]:
df = pd.read_csv('/content/heart_disease_uci.csv')

In [14]:
df.head()

,id,age,sex,dataset,cp,trestbps,chol,fbs,restecg,thalch,exang,oldpeak,slope,ca,thal,num
0,1,63,Male,Cleveland,typical angina,145.0,233.0,True,lv hypertrophy,150.0,False,2.3,downsloping,0.0,fixed defect,0
1,2,67,Male,Cleveland,asymptomatic,160.0,286.0,False,lv hypertrophy,108.0,True,1.5,flat,3.0,normal,2
2,3,67,Male,Cleveland,asymptomatic,120.0,229.0,False,lv hypertrophy,129.0,True,2.6,flat,2.0,reversable defect,1
3,4,37,Male,Cleveland,non-anginal,130.0,250.0,False,normal,187.0,False,3.5,downsloping,0.0,normal,0
4,5,41,Female,Cleveland,atypical angina,130.0,204.0,False,lv hypertrophy,172.0,False,1.4,upsloping,0.0,normal,0


In [15]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(df.iloc[:,0:-1],df.iloc[:,-1],test_size=0.2,random_state=2)

In [16]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(df.iloc[:,0:-1],df.iloc[:,-1],test_size=0.2,random_state=2)

In [17]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier

In [18]:
clf1 = LogisticRegression()
clf2 = DecisionTreeClassifier()

In [21]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.pipeline import Pipeline

# Identify numerical and categorical columns
num_cols = X_train.select_dtypes(include=['int64', 'float64']).columns.tolist()
cat_cols = X_train.select_dtypes(include=['object', 'category']).columns.tolist()

# Define preprocessing steps for numerical and categorical data
num_transformer = SimpleImputer(strategy='median')
cat_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))
])

# Combine preprocessing steps
preprocessor = ColumnTransformer(
    transformers=[
        ('num', num_transformer, num_cols),
        ('cat', cat_transformer, cat_cols)
    ]
)

# Apply preprocessing to X_train and X_test
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

# Fit the classifiers on the processed numerical data
clf1.fit(X_train_processed, y_train)
clf2.fit(X_train_processed, y_train)

/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


DecisionTreeClassifier()

In [23]:
y_pred1 = clf1.predict(X_test_processed)
y_pred2 = clf2.predict(X_test_processed)

In [28]:
from sklearn.metrics import accuracy_score

print("Logistic Regression Accuracy:", accuracy_score(y_test, y_pred1))
print("Decision Tree Accuracy:", accuracy_score(y_test, y_pred2))

Logistic Regression Accuracy: 0.5217391304347826
Decision Tree Accuracy: 0.5706521739130435


In [26]:
confusion_matrix(y_test,y_pred1)

array([[62, 13,  2,  2,  0],
       [21, 29,  7,  3,  0],
       [ 9,  8,  3,  2,  0],
       [ 3,  8,  5,  2,  0],
       [ 1,  2,  2,  0,  0]])

In [29]:
print("Logistic Regression Confusion Matrix\n")
cm = confusion_matrix(y_test, y_pred1)
pd.DataFrame(cm, columns=list(range(cm.shape[1])), index=list(range(cm.shape[0])))

Logistic Regression Confusion Matrix



,0,1,2,3,4
0,62,13,2,2,0
1,21,29,7,3,0
2,9,8,3,2,0
3,3,8,5,2,0
4,1,2,2,0,0


In [33]:
print("Decision Tree Confusion Matrix\n")
cm2 = confusion_matrix(y_test, y_pred2)
pd.DataFrame(cm2, columns=list(range(cm2.shape[1])), index=list(range(cm2.shape[0])))

Decision Tree Confusion Matrix



,0,1,2,3,4
0,63,9,5,1,1
1,10,31,6,9,4
2,3,6,5,7,1
3,1,7,2,5,3
4,0,3,0,1,1


In [42]:
result = pd.DataFrame()
result['Actual Label'] = y_test.values
result['Logistic Regression Prediction'] = y_pred1
result['Decision Tree Prediction'] = y_pred2
result.head()

,Actual Label,Logistic Regression Prediction,Decision Tree Prediction
0,0,0,2
1,0,0,0
2,1,0,1
3,0,0,0
4,0,0,0


In [36]:
result.sample(10)

,Actual Label,Logistic Regression Prediction,Decision Tree Prediction
174,0,1,0
2,1,0,1
44,0,0,0
90,0,3,0
71,0,0,0
146,1,2,3
70,0,1,1
179,1,3,1
156,0,0,1
160,3,1,4


In [43]:
from sklearn.metrics import recall_score, precision_score, f1_score

In [44]:
print("For Logistic regression Model")
print("-"*50)
cm_lr = confusion_matrix(y_test, y_pred1)
cdf = pd.DataFrame(cm_lr, columns=list(range(cm_lr.shape[1])), index=list(range(cm_lr.shape[0])))
print(cdf)
print("-"*50)
print("Precision - ", precision_score(y_test, y_pred1, average='weighted'))
print("Recall - ", recall_score(y_test, y_pred1, average='weighted'))
print("F1 score - ", f1_score(y_test, y_pred1, average='weighted'))

For Logistic regression Model
--------------------------------------------------
    0   1  2  3  4
0  62  13  2  2  0
1  21  29  7  3  0
2   9   8  3  2  0
3   3   8  5  2  0
4   1   2  2  0  0
--------------------------------------------------
Precision -  0.4755136823035851
Recall -  0.5217391304347826
F1 score -  0.4938224006463667


/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [46]:
print("For DT Model")
print("-"*50)
cm_dt = confusion_matrix(y_test, y_pred2)
cdf = pd.DataFrame(cm_dt, columns=list(range(cm_dt.shape[1])), index=list(range(cm_dt.shape[0])))
print(cdf)
print("-"*50)
print("Precision - ", precision_score(y_test, y_pred2, average='weighted', zero_division=0))
print("Recall - ", recall_score(y_test, y_pred2, average='weighted', zero_division=0))
print("F1 score - ", f1_score(y_test, y_pred2, average='weighted', zero_division=0))

For DT Model
--------------------------------------------------
    0   1  2  3  4
0  63   9  5  1  1
1  10  31  6  9  4
2   3   6  5  7  1
3   1   7  2  5  3
4   0   3  0  1  1
--------------------------------------------------
Precision -  0.588993499674029
Recall -  0.5706521739130435
F1 score -  0.5784433064893445


In [47]:
precision_score(y_test,y_pred1,average=None)

/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


array([0.64583333, 0.48333333, 0.15789474, 0.22222222, 0.        ])

In [48]:
precision_score(y_test,y_pred2,average=None)

array([0.81818182, 0.55357143, 0.27777778, 0.2173913 , 0.1       ])

In [49]:
recall_score(y_test,y_pred2,average=None)


array([0.79746835, 0.51666667, 0.22727273, 0.27777778, 0.2       ])